# Playground Series S6E11: Predicting Credit Card Fraud

- **Metric:** ROC AUC
- **Files:** `train.csv`, `test.csv`, `sample_submission.csv` (downloaded automatically via `kagglehub`)
- **Download:** `kaggle competitions download -c playground-series-s6e11` then unzip the archive

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 50)

# One seed for everything, so the folds and the models are the same on every run
SEED = 42
N_FOLDS = 5

## 1. Load data

In [2]:
import os
import kagglehub

# Download latest version (cached after the first run)
path = kagglehub.competition_download('playground-series-s6e11')
print("Path to competition files:", path)
print(os.listdir(path))

100%|██████████| 29.5M/29.5M [00:01<00:00, 16.0MB/s]

Extracting files...
Path to competition files: /Users/karenpoghosyan/.cache/kagglehub/competitions/playground-series-s6e11
['test.csv', 'train.csv', 'sample_submission.csv']


In [3]:
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_sub = pd.read_csv(os.path.join(path, "sample_submission.csv"))

print("train:", train.shape, "| test:", test.shape)
print("sample_submission columns:", list(sample_sub.columns))
train.head()

train: (699960, 26) | test: (299984, 25)
sample_submission columns: ['id', 'is_fraud']


,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,merchant_risk_score,prior_disputes,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,17,3,29.2,0,Groceries,Mastercard,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,23,5,29.9,1,Online Retail,Visa,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,15,0,24.1,0,Utilities,Visa,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,7,3,41.9,0,Groceries,Mastercard,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,4,4,48.0,0,Electronics,Visa,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


## 2. Quick EDA

The code assumes the target is the last column of `sample_submission.csv` and the ID is the first. Check this against the Data tab on Kaggle.

In [4]:
TARGET = sample_sub.columns[-1]
ID_COL = sample_sub.columns[0]
print("Target column:", TARGET, "| ID column:", ID_COL)

print(train[TARGET].value_counts(normalize=True))
print()
print(train.dtypes.value_counts())
print()
print("Missing values (top 10):")
print(train.isna().sum().sort_values(ascending=False).head(10))

Target column: is_fraud | ID column: id
is_fraud
0.0    0.9833
1.0    0.0167
Name: proportion, dtype: float64

int64      8
float64    7
bool       6
str        5
Name: count, dtype: int64

Missing values (top 10):
id                              0
amount_usd                      0
is_ai_generated_scam_attempt    0
billing_shipping_mismatch       0
ip_country_mismatch             0
used_vpn                        0
is_new_merchant                 0
is_foreign_transaction          0
device_type                     0
channel                         0
dtype: int64


## 3. Baseline: XGBoost with 5-fold stratified CV

The out-of-fold ROC AUC is the main score. The public leaderboard is only a sanity check.

In [ ]:
X = train.drop(columns=[TARGET, ID_COL], errors="ignore")
y = train[TARGET].astype(int)
X_test = test[X.columns].copy()

# Treat text columns as categories, using the same categories in train and test
cat_cols = X.select_dtypes(include=["object", "string"]).columns
for c in cat_cols:
    cats = pd.api.types.union_categoricals(
        [X[c].astype("category"), X_test[c].astype("category")]
    ).categories
    X[c] = pd.Categorical(X[c], categories=cats)
    X_test[c] = pd.Categorical(X_test[c], categories=cats)

# XGBoost wants numeric or categorical columns, so cast booleans to 0/1
bool_cols = X.select_dtypes(include="bool").columns
X[bool_cols] = X[bool_cols].astype(int)
X_test[bool_cols] = X_test[bool_cols].astype(int)

# Build the folds once and reuse them for every model, so CV scores are comparable
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

params = dict(
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    enable_categorical=True,
    eval_metric="auc",
    early_stopping_rounds=100,
    n_jobs=-1,
    random_state=SEED,
)

fold_aucs = []
for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    model = xgb.XGBClassifier(**params)
    model.fit(
        X.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
        verbose=False,
    )
    oof_xgb[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    test_xgb += model.predict_proba(X_test)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y.iloc[va_idx], oof_xgb[va_idx])
    fold_aucs.append(fold_auc)
    print(f"Fold {fold}: AUC = {fold_auc:.5f} | best iter = {model.best_iteration}")

print(f"\nMean fold AUC: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}")
print(f"OOF ROC AUC:   {roc_auc_score(y, oof_xgb):.5f}")

## 4. CatBoost on the same folds

CatBoost handles the text columns itself (`cat_features`), so it gets them as plain strings. On its own it scores about the same as XGBoost, but its predictions are different enough that blending the two helps.

In [ ]:
# CatBoost wants the categorical columns as strings
X_cb = X.copy()
X_test_cb = X_test.copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(str)
    X_test_cb[c] = X_test_cb[c].astype(str)

cb_params = dict(
    iterations=4000,
    learning_rate=0.06,
    depth=6,
    eval_metric="AUC",
    od_type="Iter",
    od_wait=200,
    random_seed=SEED,
    thread_count=-1,
    cat_features=list(cat_cols),
    verbose=False,
)

oof_cb = np.zeros(len(X))
test_cb = np.zeros(len(X_test))
for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    model = CatBoostClassifier(**cb_params)
    model.fit(
        X_cb.iloc[tr_idx], y.iloc[tr_idx],
        eval_set=(X_cb.iloc[va_idx], y.iloc[va_idx]),
        use_best_model=True,
    )
    oof_cb[va_idx] = model.predict_proba(X_cb.iloc[va_idx])[:, 1]
    test_cb += model.predict_proba(X_test_cb)[:, 1] / N_FOLDS
    fold_auc = roc_auc_score(y.iloc[va_idx], oof_cb[va_idx])
    print(f"Fold {fold}: AUC = {fold_auc:.5f} | best iter = {model.get_best_iteration()}")

print(f"\nOOF ROC AUC:   {roc_auc_score(y, oof_cb):.5f}")

## 5. Blend XGBoost and CatBoost

AUC only cares about the order of the predictions, so the blend averages ranks instead of probabilities. That way neither model dominates just because its probabilities are on a different scale. In the CV test any CatBoost weight between 0.4 and 0.6 gave nearly the same score, so the blend uses 50/50.

In [ ]:
W_CB = 0.5

def to_rank(p):
    return rankdata(p) / len(p)

oof_blend = (1 - W_CB) * to_rank(oof_xgb) + W_CB * to_rank(oof_cb)
test_blend = (1 - W_CB) * to_rank(test_xgb) + W_CB * to_rank(test_cb)

print(f"Correlation XGB vs CatBoost: {np.corrcoef(oof_xgb, oof_cb)[0, 1]:.4f}")
print(f"XGBoost  OOF AUC: {roc_auc_score(y, oof_xgb):.5f}")
print(f"CatBoost OOF AUC: {roc_auc_score(y, oof_cb):.5f}")
print(f"Blend    OOF AUC: {roc_auc_score(y, oof_blend):.5f}")
for fold, (_, va_idx) in enumerate(folds, 1):
    print(f"  Fold {fold}: blend AUC = {roc_auc_score(y.iloc[va_idx], oof_blend[va_idx]):.5f}")

## 6. Save submission

In [6]:
submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_blend})
submission.to_csv("submission.csv", index=False)
print(submission.shape)
submission.head()

(299984, 2)


,id,is_fraud
0,699960,0.000357
1,699961,0.000223
2,699962,0.085177
3,699963,0.001716
4,699964,0.001548


## Next steps

- Tune XGBoost and CatBoost (for example with Optuna)
- Add LightGBM to the blend
- Already tried without gain: merged columns, target encoding and frequency encoding